# Install Guide

This guide considers some of the most commonly encountered install issues, but not all install issues.  
If you still have install issues after this guide then please contact us.
First of all you should try to install prstools the easy way by running the following command (It assumes you have Python 3.8 or later properly installed).

```sh
pip install -U --prefer-binary prstools 
```
After this you should be able to run the command `prst` which should display the prstools help. 
If you see the help message prstools is probably properly installed and you can move on to {doc}`Getting Started Tutorial </tutorials/getting_started>` part of the tutorial. If you do not see the help message then you probably have an install issue.
    
A lot of install issues are caused by Python 3.8 or later not being installed propely.
You can see the python version in your current environment by running `python -V`, and it will not work if you do not have python at all. Also `pip` comes included with a properly installed Python 3.8+.
If your enviroment does not have the correct python you need to install it. 
Mind that using already present python installations, which are often not properly maintained, can lead to issues that are difficult to fix. The same goes for frameworks like `conda`.
You can try using already-present installations, but it's best to have a Python installation you have full control over. You do not need "sudo" or admin rights to install Python or `conda`.
To install python check out the 'Install Python' section.

Now, if the python version is sufficient and you also have `pip` potential issues with `pip install -U --prefer-binary prstools` are discussed in sections below.  
Also if you were able to install `prstools`, but the command does not work, the issue is 
probably that your path does not include everything for python to work properly. This issue & solution is discussed in a section below too.

## Install Python

There are several ways to install Python, but a great way to do it is using `conda`. Another option is using Python virtual environments with `pyenv`.

### Use `conda` to install Python (preferred)

A great benefit from having a working `conda` installation is that you can use it to install a lot of other software (samtools, compilers for C++, Java, CUDA)
with minimal effort. If you know your `conda` is working as expected you can move on to the next section.
There might already be some partially functional conda on your system, but you can install it yourself properly without admin/sudo (more on how below). With regards to `conda` already present (type `where conda` for its location) the environments can be a mixed bag (for instance if `conda` is in /usr/bin/ or in /apps/somewhere-here/ or comes via commands like `module load something`). There are many instances in which these environments are not well maintained/configured, but if you really know what you are doing you can try this. It is generally best to have your own `conda` install if you want a stable and reliable system. 
Mind, if your organization has decided to block parts of the conda ecosystem this can make things more difficult (ifso you'll probably get "ReadTimeoutError" messages).
However, there are solutions which will be discussed in the issues section.

#### Install `conda`
<!-- [here (from genome.au.dk)](https://genome.au.dk/docs/installing-software/#software-installation-with-conda) -->
Conda can be installed using [Miniforge](https://github.com/conda-forge/miniforge), following the approach:

```sh
cd~; wget https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh -O miniforge.sh
chmod +x miniforge.sh
bash miniforge.sh -b # Run the actual installation of conda
./miniforge3/bin/conda init bash; bash # Inititalize the bash environment
conda config --set channel_priority strict # Desirable configuration
```

If subsequent `conda create` or `conda install` fails it could be because Anaconda endpoints are blocked, see **Conda endpoints blocked** below.

#### Install Python using `conda`

With a proper `conda` you should be able to run `pip install -U --prefer-binary prstools` which will install prstools in your conda `base` environment.
Your command prompt should start with something like `(base)` if conda is working. You can also create and activate a new conda environment with a new Python,
by typing:

```sh
conda create -n prst python; conda activate prst
```
After this you should be able to run `pip install -U --prefer-binary prstools` without issues.

### Install Python with `pyenv`

If you do not already have a suitable Python installation, you can install one using [`pyenv`](https://github.com/pyenv/pyenv):

```bash
curl -fsSL https://pyenv.run | bash
```

Follow the shell setup instructions printed by the installer, restart your shell, then install a recent Python version with:

```bash
pyenv install 3
pyenv global 3
```

Then you can install `prstools`:

```bash
pip install -U --prefer-binary prstools
```

## Issues & solutions

### `conda create` not working:
Common Issues & fixes:
- Install hangs with "ReadTimeoutError": This probably means anaconda.org domains are blocked by your firewall, fixes discussed in "conda endpoints blocked" section below.

### Issue: `pip install` not working.

Common Issues & fixes:
- Make sure Python 3.8 or later is properly installed, which can be checked by typing `pip -V` (pip is always included with proper install). If not goto 'Install Python' section.
- Some dependencies do not install properly (e.g. h5py). You can install them seperately using `pip install h5py --only-binary=:all:` or `conda install h5py`.
- Adding `--prefer-binary` is done to have the pip installer prefer packages that have compiled binaries, meaning it will not try to compile things itself, which can give issues
  on systems with badly maintained compilers (e.g. `gcc` is old/incomplete).

### Issue: `pip install` worked, but `prstools` command not found.

It is almost certainly the case that your path variable does not include everything for python to work properly.
When there is a mixup with native pip package manager and conda, running `pip uninstall prstools; pip install -U --prefer-binary prstools ` can fix issues.
If this does not fix the issue running the command `pip uninstall prstools; pip install -U --prefer-binary prstools` successfully will give you a warning, with something like:

```WARNING: The scripts prst and prstools are installed in '/home/YOURUSERNAME/.local/bin' which is not on PATH.```

You need to add the location you see in the place of `/home/YOURUSERNAME/.local/bin` in your path. Ask a chatbot how to do this, perferably in "~/.bashrc".


### I decided to use a pre-installed Python and/or Conda

Often this means you will be using a system Python, which is generally old. Make sure it is at least version 3.8, but preferable a later version (3.10 or later).


### Issue: `conda` endpoints blocked

Some organizations block Anaconda's package servers. This can make `conda create` or `conda install` fail even when you think you have switched away from the default channels.

The simplest workaround is to explicitly select an accessible channel and tell Conda to ignore all configured/default channels:

```sh
conda create -n prst --override-channels -c https://prefix.dev/conda-forge python
conda activate prst
```

Likewise, for installing packages:

```sh
conda install --override-channels -c https://prefix.dev/conda-forge <package>
```

`--override-channels` is important here: it prevents Conda from silently adding channels inherited from configuration files elsewhere on the system.

#### Permanently removing blocked channels

If you want Conda to work normally without specifying `--override-channels`, you may need to track down and remove old channel configuration.

Start with:

```sh
conda config --show-sources
conda config --show channels default_channels channel_alias
```

Look for entries containing:

```text
defaults
repo.anaconda.com
conda.anaconda.org
anaconda.org
```

Conda can read configuration from several locations, and finding the offending setting can sometimes be surprisingly tricky. Depending on the installation, configuration may exist in places such as:

```text
~/.condarc
~/.conda/
~/.config/conda/
/etc/conda/
/etc/conda/.condarc
<conda-install>/.condarc
<environment>/.condarc
```

There may also be administrator-provided or system-wide configuration files outside your home directory. 
The important point is that Conda can combine settings from multiple configuration sources. Fixing only `~/.condarc` therefore may not be sufficient.
With some help from an LLM it is probably doable to track them all down and fix them.

Also note that:

```yaml
channels:
  - conda-forge
```

does **not** avoid Anaconda infrastructure: the normal `conda-forge` channel name resolves through `conda.anaconda.org`.

If Anaconda endpoints are blocked, configure an explicit mirror instead, for example:

```yaml
channels:
  - https://prefix.dev/conda-forge
channel_priority: strict
```

Then verify the effective configuration again:

```sh
conda config --show-sources
conda config --show channels default_channels channel_alias
```

If Conda still attempts to contact a blocked endpoint and you cannot determine where it is being configured, use `--override-channels`. This bypasses inherited channel configuration and is generally the most reliable solution on centrally managed systems.